# A6 — Persistent Homology (Topological Data Analysis)

**Goal:** Understand what persistent homology captures about painting texture — visualise persistence diagrams and barcodes, build smooth summary curves, and compare Manet vs a contemporary.

**Key papers:**
- Munnangi & Giunti (2024), arXiv:2511.16695 — *The Persistence of Painting Styles* — cross-movement discrimination, authentic vs AI-generated
- Chung, Hull & Lawson (2020), CVPR Workshops — Gaussian persistence curves for texture

**Why this is novel:** Captures topological structure of local intensity neighbourhoods — the "shape" of texture — invisible to Fourier and wavelet analysis. Also distinguishes authentic works from AI-generated fakes in an artist's style.

---

### Mathematics

We build a **Vietoris-Rips filtration** on a point cloud of local image patch feature vectors.

For each threshold $\varepsilon$, connect all point pairs within distance $\varepsilon$ to form a simplicial complex. Track topological features (connected components, loops, voids) as they **are born and die** as $\varepsilon$ increases:

- $\beta_0$: connected components (H₀)
- $\beta_1$: 1-cycles / loops (H₁)

Each feature produces a **persistence pair** $(\varepsilon_{\text{birth}}, \varepsilon_{\text{death}})$.  
**Persistence** = lifetime = $\varepsilon_{\text{death}} - \varepsilon_{\text{birth}}$.  
Long-lived features = signal. Short-lived = noise.

**Gaussian persistence curve** (Chung et al. 2020): Smooth KDE summary of the persistence diagram — directly usable as a feature vector.

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from notebooks.research.utils import load_image, to_gray, show_pair

try:
    import ripser
    from persim import plot_diagrams, PersImage
    print('ripser + persim loaded ✓')
except ImportError:
    print('Install: pip install ripser persim')
    print('(ripser is part of giotto-tda, or install separately)')
    raise

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
PATH_MANET        = Path('../../data/manet/manet_example.tif')
PATH_CONTEMPORARY = Path('../../data/contemporary/contemporary_example.tif')
PATCH_SIZE = 8     # Local patch size for point cloud construction
STRIDE     = 8     # Stride between patches (non-overlapping = PATCH_SIZE)
MAX_POINTS = 500   # Subsample point cloud for speed (Rips is O(n³))
# ─────────────────────────────────────────────────────────────────────────────

img_m  = load_image(PATH_MANET,       max_size=512)   # Smaller for TDA speed
img_c  = load_image(PATH_CONTEMPORARY, max_size=512)
gray_m = to_gray(img_m)
gray_c = to_gray(img_c)
show_pair(img_m, img_c)

---
## 1. Build Point Cloud from Image Patches

Each non-overlapping $p \times p$ patch becomes a point in $\mathbb{R}^{p^2}$ space (flattened pixel intensities). The point cloud represents the "distribution of local texture types" in the painting.

Persistent homology of this point cloud captures the **topological structure** of that distribution — e.g., whether textures cluster in rings, blobs, or more complex shapes.

In [ ]:
def image_to_point_cloud(gray: np.ndarray, patch_size: int, stride: int,
                          max_points: int = None) -> np.ndarray:
    """
    Extract non-overlapping patches as points in R^{patch_size²}.
    Normalise each patch to zero mean, unit std.
    """
    H, W = gray.shape
    points = []
    for r in range(0, H - patch_size + 1, stride):
        for c in range(0, W - patch_size + 1, stride):
            patch = gray[r:r+patch_size, c:c+patch_size].ravel()
            std = patch.std()
            if std > 1e-3:  # Skip flat patches
                points.append((patch - patch.mean()) / std)
    points = np.array(points)

    if max_points is not None and len(points) > max_points:
        idx = np.random.default_rng(42).choice(len(points), max_points, replace=False)
        points = points[idx]

    return points


pc_m = image_to_point_cloud(gray_m, PATCH_SIZE, STRIDE, MAX_POINTS)
pc_c = image_to_point_cloud(gray_c, PATCH_SIZE, STRIDE, MAX_POINTS)

print(f'Manet point cloud:        {pc_m.shape}  ({pc_m.shape[0]} points in R^{pc_m.shape[1]})')
print(f'Contemporary point cloud: {pc_c.shape}')

---
## 2. Compute Persistent Homology (Vietoris-Rips)

In [ ]:
print('Computing persistent homology (Ripser)...')
# maxdim=1 computes H₀ (connected components) and H₁ (loops)
ph_m = ripser.ripser(pc_m, maxdim=1)
ph_c = ripser.ripser(pc_c, maxdim=1)
print('Done.')

dgms_m = ph_m['dgms']   # List: dgms[0]=H₀, dgms[1]=H₁
dgms_c = ph_c['dgms']

for dim in [0, 1]:
    name_map = {0: 'H₀ (components)', 1: 'H₁ (loops)'}
    dgm_m = dgms_m[dim]
    dgm_c = dgms_c[dim]
    # Remove infinite bars (the one unpaired component in H₀)
    finite_m = dgm_m[np.isfinite(dgm_m[:, 1])]
    finite_c = dgm_c[np.isfinite(dgm_c[:, 1])]
    pers_m = finite_m[:, 1] - finite_m[:, 0]  # persistence = death - birth
    pers_c = finite_c[:, 1] - finite_c[:, 0]
    print(f'{name_map[dim]}:')
    print(f'  Manet        {len(pers_m)} features  mean persistence={pers_m.mean():.4f}  max={pers_m.max():.4f}')
    print(f'  Contemporary {len(pers_c)} features  mean persistence={pers_c.mean():.4f}  max={pers_c.max():.4f}')

---
## 3. Persistence Diagrams

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, (name, dgms) in zip(axes, [('Manet', dgms_m), ('Contemporary', dgms_c)]):
    plot_diagrams(dgms, ax=ax, show=False)
    ax.set_title(f'{name} — Persistence diagram\n'
                 'Blue=H₀ (components)  Orange=H₁ (loops)\n'
                 'Points far from diagonal = long-lived features (signal)')

plt.suptitle('Persistence diagrams: (birth, death) pairs for topological features\n'
             'Long-lived = far from the diagonal', fontsize=11)
plt.tight_layout()
plt.show()

---
## 4. Barcodes — Lifetime Visualisation

In [ ]:
def plot_barcodes(dgms, title='Barcodes', ax=None, max_bars=50):
    """Barcode plot: each bar = [birth, death] of one topological feature."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#2196F3', '#FF5722']
    y_offset = 0
    for dim, (dgm, color) in enumerate(zip(dgms[:2], colors)):
        finite = dgm[np.isfinite(dgm[:, 1])]
        # Sort by persistence (longest first)
        pers = finite[:, 1] - finite[:, 0]
        order = np.argsort(pers)[::-1][:max_bars]
        for i, idx in enumerate(order):
            b, d = finite[idx]
            ax.barh(y_offset + i, d - b, left=b, height=0.7,
                    color=color, alpha=0.7)
        y_offset += len(order) + 2

    ax.set_xlabel('Filtration value ε')
    ax.set_ylabel('Feature index')
    ax.set_title(title)
    handles = [plt.Rectangle((0,0),1,1, color='#2196F3', alpha=0.7),
               plt.Rectangle((0,0),1,1, color='#FF5722', alpha=0.7)]
    ax.legend(handles, ['H₀ (components)', 'H₁ (loops)'])
    ax.grid(alpha=0.3, axis='x')
    return ax


fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_barcodes(dgms_m, title='Manet — Barcodes\n(long bars = persistent topology)',  ax=axes[0])
plot_barcodes(dgms_c, title='Contemporary — Barcodes', ax=axes[1])
plt.tight_layout()
plt.show()

---
## 5. Gaussian Persistence Curve — Smooth Feature Vector

Convert the discrete persistence diagram to a smooth, differentiable function via **kernel density estimation** over the persistence values. This is the feature vector used for classification (Chung et al. 2020).

In [ ]:
from scipy.stats import gaussian_kde


def gaussian_persistence_curve(dgm: np.ndarray, n_points: int = 100,
                                bandwidth: float = None) -> tuple:
    """
    Gaussian KDE over persistence values: smooth summary of a persistence diagram.
    Returns (x_grid, density).
    """
    finite = dgm[np.isfinite(dgm[:, 1])]
    if len(finite) < 2:
        x = np.linspace(0, 1, n_points)
        return x, np.zeros(n_points)

    pers = finite[:, 1] - finite[:, 0]
    pers = pers[pers > 0]

    x = np.linspace(0, pers.max() * 1.1 + 1e-6, n_points)
    bw = bandwidth if bandwidth is not None else 'scott'
    kde = gaussian_kde(pers, bw_method=bw)
    return x, kde(x)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Gaussian persistence curves — smooth KDE over persistence values\n'
             'This is the feature vector used for classification', fontsize=11)

for dim, (ax, label) in enumerate(zip(axes, ['H₀ (components)', 'H₁ (loops)'])):
    x_m, kde_m = gaussian_persistence_curve(dgms_m[dim])
    x_c, kde_c = gaussian_persistence_curve(dgms_c[dim])

    # Interpolate to common grid for comparison
    x_common = np.linspace(0, max(x_m.max(), x_c.max()), 100)
    from scipy.interpolate import interp1d
    kde_m_i = interp1d(x_m, kde_m, bounds_error=False, fill_value=0)(x_common)
    kde_c_i = interp1d(x_c, kde_c, bounds_error=False, fill_value=0)(x_common)

    ax.plot(x_common, kde_m_i, color='#2196F3', lw=2, label='Manet')
    ax.plot(x_common, kde_c_i, color='#FF5722', lw=2, label='Contemporary')
    ax.fill_between(x_common, kde_m_i, kde_c_i, alpha=0.2, color='grey')
    ax.set_xlabel('Persistence (lifetime)')
    ax.set_ylabel('Density')
    ax.set_title(f'{label}')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# L2 distance between persistence curves
x_common = np.linspace(0, 1, 100)
for dim, label in [(0, 'H₀'), (1, 'H₁')]:
    x_m, kde_m = gaussian_persistence_curve(dgms_m[dim], n_points=200)
    x_c, kde_c = gaussian_persistence_curve(dgms_c[dim], n_points=200)
    # Interpolate both to [0,1]
    x_range = max(x_m.max(), x_c.max())
    x_grid = np.linspace(0, x_range, 100)
    from scipy.interpolate import interp1d
    a = interp1d(x_m, kde_m, bounds_error=False, fill_value=0)(x_grid)
    b = interp1d(x_c, kde_c, bounds_error=False, fill_value=0)(x_grid)
    l2 = np.sqrt(np.trapz((a - b)**2, x_grid))
    print(f'{label} persistence curve L2 distance: {l2:.4f}')

---
## 6. Persistence Image (alternative flat feature)

An alternative to the persistence curve: a 2D **persistence image** (Adams et al. 2017) — a pixelised representation of the (birth, persistence) plane, usable as a fixed-size feature vector.

In [ ]:
pim = PersImage(spread=0.1, pixels=[20, 20], verbose=False)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Persistence images — 20×20 pixel representation of persistence diagram\n'
             '(birth, persistence) plane, Gaussian-weighted', fontsize=11)

for row, dim in enumerate([0, 1]):
    label_str = 'H₀ (components)' if dim == 0 else 'H₁ (loops)'
    for col, (name, dgms) in enumerate([('Manet', dgms_m), ('Contemporary', dgms_c)]):
        finite = dgms[dim][np.isfinite(dgms[dim][:, 1])]
        if len(finite) > 0:
            pimg = pim.transform(finite)
            axes[row, col].imshow(pimg, cmap='viridis', origin='lower')
        axes[row, col].set_title(f'{name} — {label_str}')
        axes[row, col].set_xlabel('Birth')
        axes[row, col].set_ylabel('Persistence')

plt.tight_layout()
plt.show()

---
## 7. Key Observations

| Question | Observation |
|---|---|
| H₀: do the two paintings have different numbers of persistent components? | |
| H₁: are there meaningful loops in the texture space? What do they correspond to visually? | |
| Persistence curves: are they visually different? | |
| L2 distance: H₀ or H₁ more discriminative? | |
| Is computation time acceptable with MAX_POINTS=500? | |
| Overall: novel enough to include as Tier 3? | |

**Note:** With more paintings (N≥10 per class), build a kernel matrix $K_{ij} = \exp(-d_W(D_i, D_j)^2 / \sigma^2)$ where $d_W$ is the Wasserstein distance between persistence diagrams, and use it directly as an SVM kernel.